# DWFA — Prétraitement des données

Fichiers sources :
- `BasicAndSafelyManagedDrinkingWaterServices.csv`
- `MortalityRateAttributedToWater.csv`
- `Population.csv`
- `PoliticalStability.csv`
- `RegionCountry.csv`

## 1. Imports et chargement

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

water  = pd.read_csv("BasicAndSafelyManagedDrinkingWaterServices.csv", encoding="utf-8-sig")
mort   = pd.read_csv("MortalityRateAttributedToWater.csv", encoding="utf-8-sig")
pop    = pd.read_csv("Population.csv", encoding="utf-8-sig")
stab   = pd.read_csv("PoliticalStability.csv", encoding="utf-8-sig")
region = pd.read_csv("RegionCountry.csv", encoding="utf-8-sig")

print("water :", water.shape)
print("mort  :", mort.shape)
print("pop   :", pop.shape)
print("stab  :", stab.shape)
print("region:", region.shape)

water : (10476, 5)
mort  : (549, 5)
pop   : (20914, 4)
stab  : (3526, 4)
region: (194, 2)


## 2. Exploration rapide — valeurs manquantes et types

In [2]:
def explore(df, name):
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(df.dtypes.to_string())
    print("\nValeurs manquantes :")
    missing = df.isnull().sum()
    pct     = (missing / len(df) * 100).round(1)
    print(pd.DataFrame({"count": missing, "%": pct}).to_string())
    print(f"\nExemple :")
    display(df.head(3))

for df, name in [(water, "water"), (mort, "mort"), (pop, "pop"),
                 (stab, "stab"), (region, "region")]:
    explore(df, name)


  water
Year                                                             int64
Country                                                            str
Granularity                                                        str
Population using at least basic drinking-water services (%)    float64
Population using safely managed drinking-water services (%)    float64

Valeurs manquantes :
                                                             count     %
Year                                                             0  0.00
Country                                                          0  0.00
Granularity                                                      0  0.00
Population using at least basic drinking-water services (%)   1061 10.10
Population using safely managed drinking-water services (%)   7190 68.60

Exemple :


,Year,Country,Granularity,Population using at least basic drinking-water services (%),Population using safely managed drinking-water services (%)
0,2000,Afghanistan,Rural,21.62,NaN
1,2000,Afghanistan,Total,27.77,NaN
2,2000,Afghanistan,Urban,49.49,NaN



  mort
Year                                                               int64
Country                                                              str
Granularity                                                          str
Mortality rate attributed to exposure to unsafe WASH services    float64
WASH deaths                                                      float64

Valeurs manquantes :
                                                               count     %
Year                                                               0  0.00
Country                                                            0  0.00
Granularity                                                        0  0.00
Mortality rate attributed to exposure to unsafe WASH services      0  0.00
WASH deaths                                                      366 66.70

Exemple :


,Year,Country,Granularity,Mortality rate attributed to exposure to unsafe WASH services,WASH deaths
0,2016,Afghanistan,Female,15.31,NaN
1,2016,Afghanistan,Male,12.61,NaN
2,2016,Afghanistan,Total,13.92,4824.35



  pop
Country            str
Granularity        str
Year             int64
Population     float64

Valeurs manquantes :
             count    %
Country          0 0.00
Granularity      0 0.00
Year             0 0.00
Population       0 0.00

Exemple :


,Country,Granularity,Year,Population
0,Afghanistan,Total,2000,20779.95
1,Afghanistan,Male,2000,10689.51
2,Afghanistan,Female,2000,10090.45



  stab
Country                    str
Year                     int64
Political_Stability    float64
Granularity                str

Valeurs manquantes :
                     count    %
Country                  0 0.00
Year                     0 0.00
Political_Stability      0 0.00
Granularity              0 0.00

Exemple :


,Country,Year,Political_Stability,Granularity
0,Afghanistan,2000,-2.44,Total
1,Afghanistan,2002,-2.04,Total
2,Afghanistan,2003,-2.20,Total



  region
REGION (DISPLAY)     str
COUNTRY (DISPLAY)    str

Valeurs manquantes :
                   count    %
REGION (DISPLAY)       0 0.00
COUNTRY (DISPLAY)      0 0.00

Exemple :


,REGION (DISPLAY),COUNTRY (DISPLAY)
0,Europe,Albania
1,Europe,Andorra
2,Europe,Armenia


## 3. Nettoyage

- Renommer les colonnes
- Standardiser les types (`Year` en int, valeurs numériques en float)
- Filtrer sur `Granularity == "Total"` pour les jointures principales
- Conserver les granularités Urban/Rural séparément pour les indicateurs qui en ont besoin

In [3]:
# ── Renommage colonnes ────────────────────────────────────────────────────────
water.columns = ["annee", "pays", "granularite", "pct_acces_basique", "pct_acces_securise"]
mort.columns  = ["annee", "pays", "granularite", "taux_mortalite_wash", "wash_deaths_raw"]
pop.columns   = ["pays", "granularite", "annee", "population_milliers"]
stab.columns  = ["pays", "annee", "stabilite_politique", "granularite"]
region.columns = ["region", "pays"]

# ── Types ────────────────────────────────────────────────────────────────────
for df in [water, mort, pop, stab]:
    df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")

for col in ["pct_acces_basique", "pct_acces_securise"]:
    water[col] = pd.to_numeric(water[col], errors="coerce")

for col in ["taux_mortalite_wash", "wash_deaths_raw"]:
    mort[col] = pd.to_numeric(mort[col], errors="coerce")

pop["population_milliers"]   = pd.to_numeric(pop["population_milliers"], errors="coerce")
stab["stabilite_politique"]  = pd.to_numeric(stab["stabilite_politique"], errors="coerce")

# Plafonnement à 100 — erreurs d'arrondi source OMS (ex: 100.00001)
water["pct_acces_basique"]   = water["pct_acces_basique"].clip(upper=100)
water["pct_acces_securise"]  = water["pct_acces_securise"].clip(upper=100)

# ── Vérification doublons clés ────────────────────────────────────────────────
print("Doublons water (pays+annee+granularite) :",
      water.duplicated(["pays", "annee", "granularite"]).sum())
print("Doublons mort  (pays+annee+granularite) :",
      mort.duplicated(["pays", "annee", "granularite"]).sum())
print("Doublons pop   (pays+annee+granularite) :",
      pop.duplicated(["pays", "annee", "granularite"]).sum())
print("Doublons stab  (pays+annee)             :",
      stab.duplicated(["pays", "annee"]).sum())
print("Doublons region (pays)                  :",
      region.duplicated("pays").sum())

Doublons water (pays+annee+granularite) : 0
Doublons mort  (pays+annee+granularite) : 0
Doublons pop   (pays+annee+granularite) : 0
Doublons stab  (pays+annee)             : 0
Doublons region (pays)                  : 0


## 4. Indicateurs dérivés — population urbaine & décès WASH

In [4]:
# ── Population : pivoter urban / rural / total ────────────────────────────────
pop_pivot = (
    pop[pop["granularite"].isin(["Total", "Urban", "Rural"])]
    .pivot_table(index=["pays", "annee"], columns="granularite",
                 values="population_milliers", aggfunc="first")
    .reset_index()
)
pop_pivot.columns.name = None
pop_pivot.rename(columns={"Total": "pop_total", "Urban": "pop_urban",
                           "Rural": "pop_rural"}, inplace=True)

# Population en personnes (× 1000)
pop_pivot["nb_pop_total"]   = pop_pivot["pop_total"] * 1000
pop_pivot["nb_pop_urbaine"] = pop_pivot["pop_urban"] * 1000
pop_pivot["nb_pop_rurale"]  = pop_pivot["pop_rural"] * 1000

# % population urbaine / rurale
pop_pivot["pct_pop_urbaine"] = (pop_pivot["pop_urban"] / pop_pivot["pop_total"] * 100).round(2)
pop_pivot["pct_pop_rurale"]  = (pop_pivot["pop_rural"] / pop_pivot["pop_total"] * 100).round(2)

print(f"pop_pivot : {pop_pivot.shape}")
display(pop_pivot.head(3))

pop_pivot : (4430, 10)


,pays,annee,pop_rural,pop_total,pop_urban,nb_pop_total,nb_pop_urbaine,nb_pop_rurale,pct_pop_urbaine,pct_pop_rurale
0,Afghanistan,2000,15657.47,20779.95,4436.28,20779953.00,4436282.00,15657474.00,21.35,75.35
1,Afghanistan,2001,16318.32,21606.99,4648.14,21606988.00,4648139.00,16318324.00,21.51,75.52
2,Afghanistan,2002,17086.91,22600.77,4893.01,22600770.00,4893013.00,17086910.00,21.65,75.60


In [5]:
# ── Mortalité WASH : Total uniquement + nb décès calculé ─────────────────────
mort_total = mort[mort["granularite"] == "Total"].copy()

# Jointure temporaire avec pop pour calculer nb décès
mort_total = mort_total.merge(
    pop_pivot[["pays", "annee", "nb_pop_total"]],
    on=["pays", "annee"], how="left"
)
mort_total["nb_deces_wash"] = (
    mort_total["taux_mortalite_wash"] * mort_total["nb_pop_total"] / 100_000
).round(0)

print(f"mort_total : {mort_total.shape}")
print(f"nb_deces_wash manquants : {mort_total['nb_deces_wash'].isna().sum()}")
display(mort_total.head(3))

mort_total : (183, 7)
nb_deces_wash manquants : 1


,annee,pays,granularite,taux_mortalite_wash,wash_deaths_raw,nb_pop_total,nb_deces_wash
0,2016,Afghanistan,Total,13.92,4824.35,35383032.00,4926.00
1,2016,Albania,Total,0.17,4.87,2886438.00,5.00
2,2016,Algeria,Total,1.87,758.21,40551392.00,757.00


## 5. Jointure principale — fichier consolidé

In [6]:
# ── Eau : Total uniquement ────────────────────────────────────────────────────
water_total = water[water["granularite"] == "Total"][
    ["annee", "pays", "pct_acces_basique", "pct_acces_securise"]
].copy()

# ── Stabilité : Total uniquement ──────────────────────────────────────────────
stab_total = stab[stab["granularite"] == "Total"][
    ["pays", "annee", "stabilite_politique"]
].copy()

# ── Jointure principale ────────────────────────────────────────────────────────
df = (
    water_total
    .merge(pop_pivot, on=["pays", "annee"], how="left")
    .merge(stab_total, on=["pays", "annee"], how="left")
    .merge(region, on="pays", how="left")
)

# ── Ajout mortalité (2016 uniquement — left join) ─────────────────────────────
mort_cols = mort_total[["pays", "annee", "taux_mortalite_wash", "nb_deces_wash"]]
df = df.merge(mort_cols, on=["pays", "annee"], how="left")

print(f"df consolidé : {df.shape}")
print(f"\nCouverture région : {df['region'].notna().sum() / len(df) * 100:.1f}%")
print(f"Pays sans région  : {df[df['region'].isna()]['pays'].unique()[:10]}")
display(df.head(3))

df consolidé : (3492, 16)

Couverture région : 100.0%
Pays sans région  : <StringArray>
[]
Length: 0, dtype: str


,annee,pays,pct_acces_basique,pct_acces_securise,pop_rural,pop_total,pop_urban,nb_pop_total,nb_pop_urbaine,nb_pop_rurale,pct_pop_urbaine,pct_pop_rurale,stabilite_politique,region,taux_mortalite_wash,nb_deces_wash
0,2000,Afghanistan,27.77,NaN,15657.47,20779.95,4436.28,20779953.00,4436282.00,15657474.00,21.35,75.35,-2.44,Eastern Mediterranean,NaN,NaN
1,2000,Albania,87.87,49.29,1818.83,3129.24,1303.14,3129243.00,1303137.00,1818833.00,41.64,58.12,-0.54,Europe,NaN,NaN
2,2000,Algeria,89.84,NaN,12498.85,31042.24,18684.81,31042235.00,18684813.00,12498847.00,60.19,40.26,-1.43,Africa,NaN,NaN


## 6. Vérification de la qualité du fichier consolidé

## 6. Tests de validation

In [7]:
errors = []

# ── Volume ────────────────────────────────────────────────────────────────────
assert df["pays"].nunique() == 194,          f"ERREUR : {df['pays'].nunique()} pays au lieu de 194"
assert df["annee"].nunique() == 18,          f"ERREUR : {df['annee'].nunique()} années au lieu de 18"
assert df["region"].nunique() == 6,          f"ERREUR : {df['region'].nunique()} régions au lieu de 6"
assert len(df) == 3492,                      f"ERREUR : {len(df)} lignes au lieu de 3492"

# ── Pas de doublons ───────────────────────────────────────────────────────────
doublons = df.duplicated(["pays", "annee"]).sum()
assert doublons == 0, f"ERREUR : {doublons} doublons pays+annee"

# ── Pas de pays sans région ───────────────────────────────────────────────────
sans_region = df["region"].isna().sum()
assert sans_region == 0, f"ERREUR : {sans_region} pays sans région"

# ── Pourcentages accès eau entre 0 et 100 ────────────────────────────────────
for col in ["pct_acces_basique", "pct_acces_securise"]:
    hors_borne = df[col].dropna()
    assert ((hors_borne >= 0) & (hors_borne <= 100)).all(), f"ERREUR : {col} hors borne [0, 100]"

# ── Populations positives ─────────────────────────────────────────────────────
for col in ["nb_pop_total", "nb_pop_urbaine", "nb_pop_rurale"]:
    negatif = (df[col].dropna() < 0).sum()
    assert negatif == 0, f"ERREUR : {col} a {negatif} valeurs négatives"

# ── Avertissement : pct_pop > 100 (travailleurs migrants — source FAO) ────────
for col in ["pct_pop_urbaine", "pct_pop_rurale"]:
    hors = df[df[col] > 100][["pays", "annee", col]].dropna()
    if len(hors):
        pays_concernes = hors["pays"].unique()
        print(f"⚠ {col} > 100 pour {len(hors)} lignes — pays : {pays_concernes}")
        print("  Cause : travailleurs migrants > population résidente (source FAO). Pas un bug.")

# ── Valeurs de référence (Afghanistan) ───────────────────────────────────────
afg2000 = df[(df["pays"] == "Afghanistan") & (df["annee"] == 2000)].iloc[0]
assert abs(afg2000["pct_acces_basique"] - 27.77) < 0.01,      f"ERREUR : pct_acces_basique Afghanistan 2000 = {afg2000['pct_acces_basique']}"
assert abs(afg2000["stabilite_politique"] - (-2.44)) < 0.01,  f"ERREUR : stabilite_politique Afghanistan 2000 = {afg2000['stabilite_politique']}"

afg2016 = df[(df["pays"] == "Afghanistan") & (df["annee"] == 2016)].iloc[0]
assert abs(afg2016["nb_deces_wash"] - 4926) < 10,             f"ERREUR : nb_deces_wash Afghanistan 2016 = {afg2016['nb_deces_wash']}"

# ── Mortalité uniquement en 2016 ──────────────────────────────────────────────
annees_mort = df[df["nb_deces_wash"].notna()]["annee"].unique()
assert list(annees_mort) == [2016], f"ERREUR : nb_deces_wash présent sur d'autres années : {annees_mort}"

print("✅ Tous les tests passent — données valides.")

⚠ pct_pop_urbaine > 100 pour 49 lignes — pays : <StringArray>
['Kuwait', 'Singapore', 'Nauru', 'Monaco']
Length: 4, dtype: str
  Cause : travailleurs migrants > population résidente (source FAO). Pas un bug.
⚠ pct_pop_rurale > 100 pour 4 lignes — pays : <StringArray>
['Eritrea']
Length: 1, dtype: str
  Cause : travailleurs migrants > population résidente (source FAO). Pas un bug.
✅ Tous les tests passent — données valides.


In [8]:
missing = df.isnull().sum()
pct     = (missing / len(df) * 100).round(1)
summary = pd.DataFrame({"valeurs manquantes": missing, "%": pct})
print(summary.to_string())

print(f"\nNombre de pays distincts : {df['pays'].nunique()}")
print(f"Années couvertes         : {sorted(df['annee'].dropna().unique())}")
print(f"Régions présentes        : {df['region'].dropna().unique()}")

                     valeurs manquantes     %
annee                                 0  0.00
pays                                  0  0.00
pct_acces_basique                    43  1.20
pct_acces_securise                 1747 50.00
pop_rural                            54  1.50
pop_total                            54  1.50
pop_urban                            54  1.50
nb_pop_total                         54  1.50
nb_pop_urbaine                       54  1.50
nb_pop_rurale                        54  1.50
pct_pop_urbaine                      54  1.50
pct_pop_rurale                       54  1.50
stabilite_politique                 323  9.20
region                                0  0.00
taux_mortalite_wash                3309 94.80
nb_deces_wash                      3310 94.80

Nombre de pays distincts : 194
Années couvertes         : [np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009)

## 7. Export

Deux exports :
- `dwfa_consolidated.csv` — fichier principal pour Tableau (une ligne = un pays × une année)
- `dwfa_water_granular.csv` — accès eau avec granularité Urban/Rural (pour les graphiques Domaine 1)

In [9]:
col_order = [
    "pays", "region", "annee",
    "pct_acces_basique", "pct_acces_securise",
    "nb_pop_total", "nb_pop_urbaine", "nb_pop_rurale",
    "pct_pop_urbaine", "pct_pop_rurale",
    "stabilite_politique",
    "taux_mortalite_wash", "nb_deces_wash",
]
df_export = df[col_order].sort_values(["pays", "annee"]).reset_index(drop=True)
df_export.to_csv("dwfa_consolide.csv", index=False, encoding="utf-8-sig")
print(f"dwfa_consolide.csv exporté — {df_export.shape}")

# Granularité Urban/Rural pour l'accès eau
water_granulaire = (
    water[water["granularite"].isin(["Urban", "Rural"])]
    .merge(region, on="pays", how="left")
    [["annee", "pays", "region", "granularite", "pct_acces_basique", "pct_acces_securise"]]
    .sort_values(["pays", "annee", "granularite"])
    .reset_index(drop=True)
)
water_granulaire.to_csv("dwfa_eau_granulaire.csv", index=False, encoding="utf-8-sig")
print(f"dwfa_eau_granulaire.csv exporté — {water_granulaire.shape}")

dwfa_consolide.csv exporté — (3492, 13)


dwfa_eau_granulaire.csv exporté — (6984, 6)
